<img src="https://static.igem.wiki/teams/6333/wiki/illustrations/logos/sharp/logo.svg" height="200" align="right" style="height:240px">

# S(H)ARP

**S(H)ARP** — *Streptomyces* Hidden Antibiotic Regulated Pathways

Easy-to-use discovery of silent biosynthetic pathways in *Streptomyces* genomes. S(H)ARP reads a genome, follows the regulators that switch each pathway on or off, and maps candidate hidden pathways from end to end — pointing to targets you can later wake up with CRISPR activation.

Annotation is generated with [Bakta](https://github.com/oschwengers/bakta), motif scanning with [FIMO / MEME Suite](https://meme-suite.org/), and domain and neighborhood analysis through the [Rotifer](https://github.com/leepusp/rotifer) pipeline.

For more details, see the [S(H)ARP project wiki](https://2026.igem.wiki/usp-brazil/) and the team [background page](https://2026.igem.wiki/usp-brazil/background).

---

**Instructions**

1. Upload your input in **Cell 1** — a genome FASTA (genome-only mode, annotation runs automatically) or an annotated genome package (FASTA + GFF/GenBank + protein FASTA).
2. Run **Cell 2** to execute the analysis and download the results (`.zip` with tables and an HTML report).

Runtime note: genome-only mode installs Bakta and its database on first run, which takes several minutes. Later runs in the same session reuse what is already prepared.

---

*Developed by [iGEM USP-Brazil 2026](https://2026.igem.wiki/usp-brazil/). Contact: igem.uspbrasil@usp.br · [@igemuspbr](https://www.instagram.com/igemuspbr)*


In [ ]:
# @title S(H)ARP — Setup, upload input, and prepare backend { display-mode: "form" }
# @markdown Prepare Colab runtime, clone GitLab S(H)ARP + Rotifer, upload files, infer input mode, and prepare Bakta only when needed.

JOB_NAME = "sharp_run_01" # @param {type:"string"}
ORGANISM_NAME = "Streptomyces sp." # @param {type:"string"}
STRAIN_NAME = "unknown" # @param {type:"string"}
SHOW_SETUP_LOGS = True # @param {type:"boolean"}

from pathlib import Path
from functools import lru_cache
import os
import sys
import re
import json
import time
import shutil
import hashlib
import subprocess
import zipfile
import tarfile
import gzip
import urllib.request
import importlib

os.environ["MPLBACKEND"] = "Agg"

# ============================================================
# Configuration
# ============================================================

SHARP_PROJECT_NAME = "sharp_igem_usp_brazil_2026"

SHARP_REPO_URL = "https://gitlab.igem.org/2026/software/usp-brazil/sharp.git"
SHARP_BRANCH = "main"

ROTIFER_REPO_URL = "https://github.com/leepusp/rotifer.git"
ROTIFER_BRANCH = "master"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

CONTENT_DIR = Path("/content") if Path("/content").exists() else Path.cwd()


def clean_name(value):
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value).strip()).strip("._-")
    return value or "sharp_run_01"


RUN_NAME = clean_name(JOB_NAME)

# ============================================================
# Runtime paths
# ============================================================

PROJECT_DIR = CONTENT_DIR / SHARP_PROJECT_NAME
DATA_DIR = PROJECT_DIR / "data"
INPUT_DIR = DATA_DIR / "input"
NORMALIZED_DIR = DATA_DIR / "normalized"
DATABASES_DIR = PROJECT_DIR / "databases"
CONFIG_DIR = PROJECT_DIR / "config"
RESULTS_DIR = PROJECT_DIR / "results"
RUN_DIR = RESULTS_DIR / RUN_NAME
LOG_DIR = RUN_DIR / "logs"
TABLE_DIR = RUN_DIR / "tables"
REPORT_DIR = RUN_DIR / "report"

ENV_DIR = PROJECT_DIR / "envs"
MEME_ENV_DIR = ENV_DIR / "meme"
BAKTA_ENV_DIR = ENV_DIR / "bakta"

SHARP_DIR = CONTENT_DIR / "sharp"
ROTIFER_DIR = CONTENT_DIR / "rotifer"
ROTIFER_LIB = ROTIFER_DIR / "lib"
ROTIFER_BIN = ROTIFER_DIR / "bin"

COMMON_READY = PROJECT_DIR / "SHARP_COMMON_READY"

for directory in [
    PROJECT_DIR, DATA_DIR, INPUT_DIR, NORMALIZED_DIR, DATABASES_DIR, CONFIG_DIR,
    RESULTS_DIR, RUN_DIR, LOG_DIR, TABLE_DIR, REPORT_DIR, ENV_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

SETUP_EVENTS = []
_T0 = time.time()

# ============================================================
# Small infrastructure helpers
# ============================================================

def log_event(message):
    SETUP_EVENTS.append({"time": time.strftime("%Y-%m-%d %H:%M:%S"), "message": str(message)})
    if SHOW_SETUP_LOGS:
        print(f"[{time.time() - _T0:6.1f}s] {message}", flush=True)


def run(command, check=True, env=None, stream=False):
    """Run a command. With stream=True the child inherits the console so
    long-running tools (conda, curl, amrfinder_update) show live progress."""
    command_env = os.environ.copy()
    command_env["MPLBACKEND"] = "Agg"
    if env:
        command_env.update(env)

    if stream:
        result = subprocess.run(
            command,
            shell=isinstance(command, str),
            text=True,
            check=False,
            env=command_env,
        )
        if check and result.returncode != 0:
            cmd = " ".join(command) if isinstance(command, list) else command
            raise RuntimeError(f"Command failed (output above):\n{cmd}\nreturncode={result.returncode}")
        return result

    result = subprocess.run(
        command,
        shell=isinstance(command, str),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
        env=command_env,
    )

    if check and result.returncode != 0:
        cmd = " ".join(command) if isinstance(command, list) else command
        raise RuntimeError(
            f"Command failed:\n{cmd}\n\n"
            f"STDOUT:\n{(result.stdout or '')[-4000:]}\n\n"
            f"STDERR:\n{(result.stderr or '')[-4000:]}"
        )

    return result


def pip_install(*packages):
    run([sys.executable, "-m", "pip", "install", "-q", *packages])


def ensure_module(module_name, pip_name=None):
    """Import a module, pip-installing it once if missing (not gated by COMMON_READY)."""
    try:
        return importlib.import_module(module_name)
    except Exception:
        pip_install(pip_name or module_name)
        importlib.invalidate_caches()
        return importlib.import_module(module_name)


def neutralize_ete3_taxonomy():
    """Rotifer eagerly loads an optional ete3 NCBI-taxonomy backend whose NCBITaxa()
    downloads a large NCBI dump on first use. S(H)ARP never uses taxonomy, so replace
    NCBITaxa with a no-op stub (before importing sharp/Rotifer) to skip the download
    while still letting Rotifer's backend import succeed."""
    try:
        from ete3.ncbi_taxonomy import ncbiquery as _nq
    except Exception:
        return False

    class _NoTaxa:
        def __init__(self, *args, **kwargs):
            pass

        def __getattr__(self, _name):
            return lambda *args, **kwargs: {}

    _nq.NCBITaxa = _NoTaxa
    return True


def prepend_path(path):
    path = Path(path)
    if path.exists() and str(path) not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = str(path) + os.pathsep + os.environ.get("PATH", "")


def prepend_pythonpath(path):
    path = Path(path)
    if not path.exists():
        return
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
    current = os.environ.get("PYTHONPATH", "")
    parts = [item for item in current.split(os.pathsep) if item]
    if str(path) not in parts:
        os.environ["PYTHONPATH"] = str(path) + (os.pathsep + current if current else "")


def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False, default=str), encoding="utf-8")


def checksum(path, algo="sha256"):
    h = hashlib.new(algo)
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024 * 8), b""):
            h.update(block)
    return h.hexdigest()


def download_file(url, output_path, stream=False):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and output_path.stat().st_size > 0:
        return "existing"

    if shutil.which("curl"):
        curl = ["curl", "-L", "--fail", "--connect-timeout", "30",
                "--retry", "5", "--retry-delay", "5"]
        curl += ["-#"] if stream else ["-s"]   # -# shows a live progress bar
        curl += [url, "-o", str(output_path)]
        run(curl, stream=stream)
    else:
        with urllib.request.urlopen(url, timeout=120) as response:
            output_path.write_bytes(response.read())

    if not output_path.exists() or output_path.stat().st_size == 0:
        raise RuntimeError(f"Downloaded file is empty: {url}")

    return url


def open_text_maybe_gzip(path):
    path = Path(path)
    if path.name.lower().endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8", errors="replace")
    return path.open("r", encoding="utf-8", errors="replace")


def copy_maybe_decompress(source_path, destination_path):
    source_path = Path(source_path)
    destination_path = Path(destination_path)
    destination_path.parent.mkdir(parents=True, exist_ok=True)

    if source_path.name.lower().endswith(".gz"):
        with gzip.open(source_path, "rb") as src, destination_path.open("wb") as dst:
            shutil.copyfileobj(src, dst)
    else:
        shutil.copy2(source_path, destination_path)

    return destination_path


@lru_cache(maxsize=None)
def fasta_stats(path):
    records = 0
    total = 0
    current = 0
    first_id = ""

    with open_text_maybe_gzip(path) as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if records:
                    total += current
                records += 1
                current = 0
                if not first_id:
                    first_id = line[1:].split()[0]
            else:
                current += len(re.sub(r"[^A-Za-z]", "", line))

    if records:
        total += current

    return {"records": records, "total_bp": total, "first_id": first_id}


def _is_within(root, target):
    try:
        Path(target).resolve().relative_to(Path(root).resolve())
        return True
    except ValueError:
        return False


def safe_extract_zip(archive_path, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path, "r") as archive:
        for member in archive.namelist():
            if not _is_within(output_dir, output_dir / member):
                raise RuntimeError(f"Unsafe ZIP member path: {member}")
        archive.extractall(output_dir)
    return [p for p in output_dir.rglob("*") if p.is_file()]


def safe_extract_tar(archive_path, output_dir, mode="r:*"):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, mode) as archive:
        for member in archive.getmembers():
            if not _is_within(output_dir, output_dir / member.name):
                raise RuntimeError(f"Unsafe TAR member path: {member.name}")
        try:
            archive.extractall(output_dir, filter="data")
        except TypeError:
            archive.extractall(output_dir)
    return [p for p in output_dir.rglob("*") if p.is_file()]


def ensure_repo(repo_url, branch, target_dir):
    target_dir = Path(target_dir)
    if (target_dir / ".git").exists():
        run(["git", "-C", str(target_dir), "fetch", "origin", branch, "--depth", "1"])
        run(["git", "-C", str(target_dir), "checkout", branch])
        run(["git", "-C", str(target_dir), "reset", "--hard", f"origin/{branch}"])
    else:
        if target_dir.exists():
            shutil.rmtree(target_dir)
        run(["git", "clone", "--depth", "1", "--branch", branch, repo_url, str(target_dir)])
    return target_dir


def ensure_mamba():
    prepend_path("/usr/local/bin")
    if shutil.which("mamba"):
        return shutil.which("mamba")

    installer = ENV_DIR / "Miniforge3-Linux-x86_64.sh"
    if not installer.exists():
        run(["wget", "-q", "-O", str(installer),
             "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh"])
    run(["bash", str(installer), "-bfp", "/usr/local"])
    run(["conda", "config", "--set", "auto_update_conda", "false"], check=False)  # mamba 2.x dropped `config`

    if not shutil.which("mamba"):
        raise RuntimeError("mamba was not found after Miniforge installation.")
    return shutil.which("mamba")


def hmmpress_local(hmm_path):
    """Create HMMER binary indexes; also enables Rotifer's fast optimized_profiles path."""
    hmm_path = Path(hmm_path)
    if not hmm_path.exists() or hmm_path.stat().st_size == 0:
        raise RuntimeError(f"HMM file not found or empty: {hmm_path}")

    indexes = [Path(str(hmm_path) + ext) for ext in (".h3f", ".h3i", ".h3m", ".h3p")]
    for index_path in indexes:
        if index_path.exists():
            index_path.unlink()

    run(["hmmpress", "-f", str(hmm_path)], check=True)

    missing = [str(p) for p in indexes if not p.exists()]
    if missing:
        raise RuntimeError("hmmpress did not create all expected index files: " + ", ".join(missing))

    return {"hmm": str(hmm_path), "indexes": [str(p) for p in indexes]}


def patch_bakta_decode(bakta_env_dir):
    """Bakta calls hit.name.decode(); current PyHMMER returns str, so guard every
    `.decode()` in the bakta package with a helper that only decodes bytes.
    Idempotent: re-running reports 'already_patched'."""
    py = Path(bakta_env_dir) / "bin" / "python"
    if not py.exists():
        py = Path(sys.executable)

    patch_script = r'''
import re, shutil, time
from pathlib import Path
import bakta

pkg = Path(bakta.__file__).parent
start_marker = "# S(H)ARP compatibility helper start"
end_marker = "# S(H)ARP compatibility helper end"

helper = (
    "# S(H)ARP compatibility helper start\n"
    "def _sharp_decode(value):\n"
    "    if hasattr(value, 'decode'):\n"
    "        return value.decode()\n"
    "    return str(value)\n"
    "# S(H)ARP compatibility helper end\n\n"
)

decode_pattern = re.compile(
    r"(?<![A-Za-z0-9_])"
    r"([A-Za-z_][A-Za-z0-9_]*(?:\.[A-Za-z_][A-Za-z0-9_]*)*)"
    r"\.decode\(\)"
)

def remove_helper(text):
    while start_marker in text and end_marker in text:
        start = text.index(start_marker)
        end = text.index(end_marker) + len(end_marker)
        if end < len(text) and text[end:end + 1] == "\n":
            end += 1
        text = text[:start] + text[end:]
    return text

def insert_helper(text):
    lines = text.splitlines()
    insert_at = 0
    for index, line in enumerate(lines):
        stripped = line.strip()
        if stripped.startswith("import ") or stripped.startswith("from "):
            insert_at = index + 1
            continue
        if insert_at > 0 and stripped and not stripped.startswith("#"):
            break
    lines.insert(insert_at, helper.rstrip("\n"))
    return "\n".join(lines) + "\n"

patched = []
already = []
for path in sorted(pkg.rglob("*.py")):
    if "__pycache__" in path.parts or ".sharp_backup_" in path.name:
        continue
    text = path.read_text(encoding="utf-8")
    if ".decode()" not in text and "_sharp_decode(" not in text:
        continue
    clean = remove_helper(text)
    new = decode_pattern.sub(r"_sharp_decode(\1)", clean)
    if "_sharp_decode(" in new:
        new = insert_helper(new)
    if new != text:
        shutil.copy2(path, path.with_suffix(path.suffix + f".sharp_backup_{int(time.time())}"))
        path.write_text(new, encoding="utf-8")
        patched.append(str(path))
    else:
        already.append(str(path))

print("bakta_patch_status=" + ("patched" if patched else "already_patched" if already else "not_needed"))
print("bakta_patch_files=" + str(len(patched) + len(already)))
'''

    return run([str(py), "-c", patch_script], check=True).stdout

# ============================================================
# Common dependencies
# ============================================================

print(f"S(H)ARP setup started: {RUN_NAME}")

for bin_dir in (MEME_ENV_DIR / "bin", BAKTA_ENV_DIR / "bin", "/usr/local/bin", "/usr/bin", "/bin"):
    prepend_path(bin_dir)

log_event("Preparing common runtime dependencies.")

if IN_COLAB and not COMMON_READY.exists():
    run("apt-get update -qq")
    run(
        "DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
        "git curl wget xz-utils bzip2 hmmer graphviz graphviz-dev pkg-config",
        check=False,
    )
    pip_install(
        "numpy", "pandas", "biopython", "bcbio-gff", "requests", "tqdm", "joblib",
        "pyhmmer", "ete3", "seaborn", "matplotlib", "networkx", "sqlalchemy",
        "openpyxl", "pyyaml",
        "legacy-cgi",  # Python 3.13 removed the stdlib `cgi` module that ete3 imports
    )
    try:
        pip_install("pygraphviz")
    except Exception:
        pass
    COMMON_READY.touch()

# Guaranteed even on a cached runtime (COMMON_READY already present):
ensure_module("cgi", "legacy-cgi")   # must exist before Rotifer/ete3 is imported
ensure_module("BCBio.GFF", "bcbio-gff")

# ============================================================
# Clone GitLab S(H)ARP package and Rotifer runtime dependency
# ============================================================

log_event("Cloning GitLab S(H)ARP package.")
ensure_repo(SHARP_REPO_URL, SHARP_BRANCH, SHARP_DIR)

log_event("Cloning Rotifer runtime dependency.")
ensure_repo(ROTIFER_REPO_URL, ROTIFER_BRANCH, ROTIFER_DIR)

prepend_pythonpath(ROTIFER_LIB)
prepend_pythonpath(SHARP_DIR)
prepend_path(ROTIFER_BIN)

# ============================================================
# Prepare FIMO/MEME
# ============================================================

log_event("Preparing FIMO/MEME.")

if not shutil.which("fimo"):
    mamba = ensure_mamba()
    if not (MEME_ENV_DIR / "bin" / "fimo").exists():
        log_event("Creating MEME/FIMO conda env (solve + download, a few minutes)...")
        run([mamba, "create", "-y", "-p", str(MEME_ENV_DIR),
             "-c", "conda-forge", "-c", "bioconda", "meme"], stream=SHOW_SETUP_LOGS)
    prepend_path(MEME_ENV_DIR / "bin")

if not shutil.which("fimo"):
    raise RuntimeError("FIMO was not found after MEME Suite installation.")

# ============================================================
# Validate official S(H)ARP package
# ============================================================

log_event("Validating S(H)ARP package.")

# Skip Rotifer's optional ete3 NCBI-taxonomy download (unused by S(H)ARP).
neutralize_ete3_taxonomy()

importlib.invalidate_caches()
sharp = importlib.import_module("sharp")
sharp_pipeline = importlib.import_module("sharp.pipeline")

if not hasattr(sharp_pipeline, "igem_pipeline"):
    raise RuntimeError("sharp.pipeline.igem_pipeline was not found.")

SHARP_PACKAGE_STATUS = {
    "import_ok": True,
    "repo_url": SHARP_REPO_URL,
    "branch": SHARP_BRANCH,
    "sharp_dir": str(SHARP_DIR),
    "sharp_file": str(Path(sharp.__file__).resolve()),
    "pipeline_file": str(Path(sharp_pipeline.__file__).resolve()),
    "pipeline_function": "sharp.pipeline.igem_pipeline",
}

# ============================================================
# Copy Colab resources from GitLab clone
# ============================================================

log_event("Preparing S(H)ARP Colab resources.")

GITLAB_COLAB_RESOURCES = SHARP_DIR / "colab" / "resources"

RESOURCE_SOURCES = {
    "bakta_light_config": GITLAB_COLAB_RESOURCES / "config" / "bakta_light_cache.json",
    "heptamer_meme": GITLAB_COLAB_RESOURCES / "motifs" / "heptarepeats2.meme",
    "sarp_hmm": GITLAB_COLAB_RESOURCES / "hmm" / "sarp_custom.hmm",
    "domain_models_hmm": GITLAB_COLAB_RESOURCES / "domain_models" / "domain_models.hmm",
    "domain_modelnames": GITLAB_COLAB_RESOURCES / "domain_models" / "hmm_modelnames.tsv",
    "sharp_config": SHARP_DIR / "config.yaml",
}

SHARP_INTERNAL_RESOURCES = {
    "bakta_light_config": str(CONFIG_DIR / "bakta_light_cache.json"),
    "heptamer_meme": str(DATABASES_DIR / "motifs" / "heptarepeats2.meme"),
    "sarp_hmm": str(DATABASES_DIR / "hmm" / "sarp_custom.hmm"),
    "domain_models_hmm": str(DATABASES_DIR / "domain_models" / "domain_models.hmm"),
    "domain_modelnames": str(DATABASES_DIR / "domain_models" / "hmm_modelnames.tsv"),
    "sharp_config": str(CONFIG_DIR / "sharp_config.yaml"),
}


def locate_in_repo(expected_path):
    """Return expected_path, or fall back to the first match of its basename in the clone."""
    expected_path = Path(expected_path)
    if expected_path.exists() and expected_path.stat().st_size > 0:
        return expected_path
    hits = sorted(SHARP_DIR.rglob(expected_path.name))
    return hits[0] if hits else None


SHARP_RESOURCE_STATUS = {}

for key, source_path in RESOURCE_SOURCES.items():
    resolved = locate_in_repo(source_path)
    if resolved is None:
        raise RuntimeError(
            f"Missing resource in GitLab S(H)ARP clone: {source_path} "
            "(expected under sharp/colab/resources/ in the repo)."
        )

    target_path = Path(SHARP_INTERNAL_RESOURCES[key])
    target_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(resolved, target_path)

    SHARP_RESOURCE_STATUS[key] = {
        "available": True,
        "source": str(resolved),
        "path": str(target_path),
        "size": target_path.stat().st_size,
        "sha256": checksum(target_path, "sha256"),
    }

# Pressing once here lets Rotifer use the fast optimized_profiles path (no runtime patch).
SHARP_HMM_PRESS_STATUS = [
    hmmpress_local(SHARP_INTERNAL_RESOURCES["domain_models_hmm"]),
    hmmpress_local(SHARP_INTERNAL_RESOURCES["sarp_hmm"]),
]

# ============================================================
# Upload input files and infer mode automatically
# ============================================================

log_event("Uploading and normalizing input files.")

for d in (INPUT_DIR, NORMALIZED_DIR):
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

uploaded_paths = []

if IN_COLAB:
    from google.colab import files
    for filename, content in files.upload().items():
        destination = INPUT_DIR / Path(filename).name
        destination.write_bytes(content)
        uploaded_paths.append(destination)
else:
    uploaded_paths = [p for p in INPUT_DIR.rglob("*") if p.is_file()]


def safe_extract(path):
    path = Path(path)
    lower = path.name.lower()
    if lower.endswith(".zip"):
        return safe_extract_zip(path, INPUT_DIR / f"{path.stem}_extracted")
    if lower.endswith((".tar", ".tar.gz", ".tgz", ".tar.xz")):
        return safe_extract_tar(path, INPUT_DIR / f"{path.name.replace('.', '_')}_extracted")
    return []


for path in list(uploaded_paths):
    uploaded_paths.extend(safe_extract(path))

all_files = [p for p in INPUT_DIR.rglob("*") if p.is_file()]

if not all_files:
    raise RuntimeError("No input files were uploaded.")

# ---- classification (single pass) --------------------------

ANNOTATION_SUFFIXES = {".gff", ".gff3", ".gb", ".gbk", ".gbff", ".genbank"}
GENBANK_SUFFIXES = {".gb", ".gbk", ".gbff", ".genbank"}
FASTA_SUFFIXES = {".fa", ".fas", ".fasta", ".fna", ".ffn", ".faa", ".pep"}


def normalized_name(path):
    name = Path(path).name.lower()
    return name[:-3] if name.endswith(".gz") else name


def suffix(path):
    return Path(normalized_name(path)).suffix.lower()


def classify(path):
    name = normalized_name(path)
    sfx = Path(name).suffix.lower()
    if sfx in ANNOTATION_SUFFIXES:
        return "annotation"
    if name.endswith((".faa", ".pep")) or "protein" in name:
        return "protein"
    if name.endswith(".ffn") or "cds" in name:
        return "cds"
    if sfx in FASTA_SUFFIXES:
        return "genome"
    return "other"


def choose_largest_fasta(paths):
    return max(paths, key=lambda p: fasta_stats(p)["total_bp"]) if paths else None


buckets = {"annotation": [], "protein": [], "cds": [], "genome": [], "other": []}
for p in all_files:
    buckets[classify(p)].append(p)

genome_candidates = buckets["genome"]
annotation_candidates = buckets["annotation"]
protein_candidates = buckets["protein"]
cds_candidates = buckets["cds"]

genome_source = choose_largest_fasta(genome_candidates)
annotation_source = annotation_candidates[0] if annotation_candidates else None
protein_source = protein_candidates[0] if protein_candidates else None
cds_source = cds_candidates[0] if cds_candidates else None

if genome_source is None:
    detected = ", ".join(p.name for p in all_files)
    raise RuntimeError(f"No genome FASTA was detected. Detected files: {detected}")

has_partial_annotation_package = annotation_source is not None or protein_source is not None

if annotation_source is not None and protein_source is not None:
    INPUT_MODE_KEY = "annotated_genome_package"
    workflow = "sharp_existing_annotation"
    annotation_mode = "use_existing_annotation"
    annotation_backend = "existing_annotation"
    requires_bakta_execution = False
    requires_bakta_db = False
elif has_partial_annotation_package:
    detected = ", ".join(p.name for p in all_files)
    raise RuntimeError(
        "Partial annotated package detected. Provide genome FASTA + annotation + protein FASTA, "
        f"or only genome FASTA for automatic Bakta annotation. Detected files: {detected}"
    )
else:
    INPUT_MODE_KEY = "genome_fasta_only"
    workflow = "sharp_auto_from_genome"
    annotation_mode = "auto_from_genome"
    annotation_backend = "bakta_cached_light_db"
    requires_bakta_execution = True
    requires_bakta_db = True

GENOME_FASTA = NORMALIZED_DIR / "sharp_input_genome.fna"
ANNOTATION_FILE = None
PROTEIN_FASTA = None
CDS_FASTA = None
genome_format = ""

copy_maybe_decompress(genome_source, GENOME_FASTA)

if INPUT_MODE_KEY == "annotated_genome_package":
    if suffix(annotation_source) in GENBANK_SUFFIXES:
        ANNOTATION_FILE = NORMALIZED_DIR / "sharp_input_annotation.gbff"
        genome_format = "gbk"
    else:
        ANNOTATION_FILE = NORMALIZED_DIR / "sharp_input_annotation.gff3"
        genome_format = "gff"

    PROTEIN_FASTA = NORMALIZED_DIR / "sharp_input_proteins.faa"
    copy_maybe_decompress(annotation_source, ANNOTATION_FILE)
    copy_maybe_decompress(protein_source, PROTEIN_FASTA)

    if cds_source is not None:
        CDS_FASTA = NORMALIZED_DIR / "sharp_input_cds.ffn"
        copy_maybe_decompress(cds_source, CDS_FASTA)

genome_stats = fasta_stats(GENOME_FASTA)
genome_hash = checksum(GENOME_FASTA, "sha256")

# ============================================================
# Prepare Bakta backend only for genome FASTA only mode
# ============================================================

def ensure_bakta_backend():
    log_event("Preparing Bakta backend.")
    prepend_path(BAKTA_ENV_DIR / "bin")

    bakta_bin = shutil.which("bakta")
    if not bakta_bin:
        mamba = ensure_mamba()
        if not (BAKTA_ENV_DIR / "bin" / "bakta").exists():
            # Dedicated env: base pins Python 3.13, bakta needs <3.12 (solver picks a compatible one).
            log_event("Creating Bakta conda env (solve + download, several minutes)...")
            run([mamba, "create", "-y", "-p", str(BAKTA_ENV_DIR),
                 "-c", "conda-forge", "-c", "bioconda", "bakta=1.11.3", "ncbi-amrfinderplus"],
                stream=SHOW_SETUP_LOGS)
        prepend_path(BAKTA_ENV_DIR / "bin")
        bakta_bin = shutil.which("bakta")

    if not bakta_bin:
        raise RuntimeError("Bakta was not found after installation.")

    # Bakta 1.11.3 assumes PyHMMER returns bytes (hit.name.decode()); patch it for str.
    log_event("Patching Bakta for current PyHMMER (.decode)...")
    log_event(patch_bakta_decode(BAKTA_ENV_DIR).strip().replace("\n", " | "))

    bakta_config_path = Path(SHARP_INTERNAL_RESOURCES["bakta_light_config"])
    if not bakta_config_path.exists():
        raise RuntimeError(f"Bakta light config not found: {bakta_config_path}")

    bakta_config = json.loads(bakta_config_path.read_text(encoding="utf-8"))
    section = bakta_config.get("bakta", bakta_config)

    archive_name = section.get("archive_name", "db-light.tar.xz")
    expected_md5 = section.get("expected_md5", "")
    expected_sha256 = section.get("expected_sha256", "")
    release_url = section.get("github_release_url", "")
    fallback_url = section.get("zenodo_fallback_url", "")

    if not release_url:
        raise RuntimeError("Bakta light config does not define github_release_url.")

    cache_dir = DATABASES_DIR / "bakta_cache"
    extract_dir = DATABASES_DIR / "bakta"
    archive = cache_dir / archive_name
    db_path = extract_dir / "db-light"
    cache_dir.mkdir(parents=True, exist_ok=True)
    extract_dir.mkdir(parents=True, exist_ok=True)

    archive_ready = archive.exists() and archive.stat().st_size > 0
    if archive_ready and expected_md5:
        archive_ready = checksum(archive, "md5") == expected_md5
    if archive_ready and expected_sha256:
        archive_ready = checksum(archive, "sha256") == expected_sha256

    if not archive_ready:
        if archive.exists():
            archive.unlink()
        log_event(f"Downloading Bakta light DB (~1.5 GB): {archive_name}")
        try:
            download_file(release_url, archive, stream=SHOW_SETUP_LOGS)
        except Exception:
            if not fallback_url:
                raise
            log_event("Primary URL failed; trying Zenodo fallback...")
            download_file(fallback_url, archive, stream=SHOW_SETUP_LOGS)

        log_event("Verifying archive checksum...")
        if expected_md5 and checksum(archive, "md5") != expected_md5:
            raise RuntimeError("Bakta light DB archive MD5 mismatch.")
        if expected_sha256 and checksum(archive, "sha256") != expected_sha256:
            raise RuntimeError("Bakta light DB archive SHA256 mismatch.")

    if not (db_path / "version.json").exists():
        log_event("Extracting Bakta light DB...")
        safe_extract_tar(archive, extract_dir, mode="r:xz")

    if not (db_path / "version.json").exists():
        version_files = sorted(extract_dir.rglob("version.json"))
        if version_files:
            db_path = version_files[0].parent

    if not (db_path / "version.json").exists():
        raise RuntimeError("Bakta light DB extraction failed: version.json not found.")

    db_version = json.loads((db_path / "version.json").read_text(encoding="utf-8"))
    os.environ["BAKTA_DB"] = str(db_path)

    amrfinder_update = shutil.which("amrfinder_update")
    if not amrfinder_update:
        raise RuntimeError("amrfinder_update was not found after Bakta installation.")

    amrfinder_db_path = db_path / "amrfinderplus-db"
    amrfinder_sentinel = amrfinder_db_path / ".sharp_amrfinder_update_ok.json"
    if not amrfinder_sentinel.exists():
        log_event("Updating AMRFinder+ DB (downloads from NCBI)...")
        run([amrfinder_update, "--force_update", "--database", str(amrfinder_db_path)], stream=SHOW_SETUP_LOGS)
        write_json(amrfinder_sentinel, {
            "status": "ready",
            "database": str(amrfinder_db_path),
            "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        })

    threads = min(2, max(1, os.cpu_count() or 2))
    runtime_args = [
        "--threads", str(threads),
        "--skip-trna", "--skip-tmrna", "--skip-rrna", "--skip-ncrna",
        "--skip-ncrna-region", "--skip-crispr", "--skip-sorf",
        "--skip-gap", "--skip-ori", "--skip-plot",
    ]

    return {
        "requires_bakta_execution": True,
        "requires_bakta_db": True,
        "bakta_bin": str(bakta_bin),
        "bakta_db_ready": True,
        "bakta_db_path": str(db_path),
        "bakta_db_version": db_version,
        "bakta_threads": threads,
        "bakta_runtime_args": runtime_args,
        "amrfinder_db_ready": True,
        "amrfinder_db_path": str(amrfinder_db_path),
        "amrfinder_sentinel": str(amrfinder_sentinel),
    }


if requires_bakta_execution:
    SHARP_BACKEND = ensure_bakta_backend()
else:
    SHARP_BACKEND = {
        "requires_bakta_execution": False,
        "requires_bakta_db": False,
        "bakta_bin": "",
        "bakta_db_ready": False,
        "bakta_db_path": "",
        "bakta_db_version": {},
        "bakta_threads": 0,
        "bakta_runtime_args": [],
        "amrfinder_db_ready": False,
        "amrfinder_db_path": "",
    }

# ============================================================
# Save context for execution cell
# ============================================================

SHARP_CONTEXT_FILE = CONFIG_DIR / "sharp_notebook_context.json"
INPUT_CONTRACT_FILE = CONFIG_DIR / "sharp_input_contract.json"
INPUT_FILES_FILE = CONFIG_DIR / "sharp_input_files.json"
RESOURCE_MANIFEST_FILE = CONFIG_DIR / "sharp_resource_manifest.json"
DEPENDENCY_MANIFEST_FILE = RUN_DIR / f"{RUN_NAME}_dependency_manifest.json"
SETUP_LOG_FILE = RUN_DIR / f"{RUN_NAME}_setup_events.json"

SHARP_PATHS = {
    "project_dir": str(PROJECT_DIR),
    "data_dir": str(DATA_DIR),
    "input_dir": str(INPUT_DIR),
    "normalized_dir": str(NORMALIZED_DIR),
    "databases_dir": str(DATABASES_DIR),
    "config_dir": str(CONFIG_DIR),
    "results_dir": str(RESULTS_DIR),
    "run_name": RUN_NAME,
    "run_dir": str(RUN_DIR),
    "log_dir": str(LOG_DIR),
    "table_dir": str(TABLE_DIR),
    "report_dir": str(REPORT_DIR),
    "sharp_dir": str(SHARP_DIR),
    "rotifer_dir": str(ROTIFER_DIR),
    "rotifer_lib": str(ROTIFER_LIB),
    "rotifer_bin": str(ROTIFER_BIN),
}

SHARP_INPUT_CONTRACT = {
    "status": "ready",
    "workflow": workflow,
    "mode": INPUT_MODE_KEY,
    "annotation_mode": annotation_mode,
    "annotation_backend": annotation_backend,
    "requires_bakta_execution": requires_bakta_execution,
    "requires_bakta_db": requires_bakta_db,
    "genome_fasta": str(GENOME_FASTA),
    "annotation_file": str(ANNOTATION_FILE) if ANNOTATION_FILE else "",
    "genome_format": genome_format,
    "protein_fasta": str(PROTEIN_FASTA) if PROTEIN_FASTA else "",
    "cds_fasta": str(CDS_FASTA) if CDS_FASTA else "",
    "organism_name": ORGANISM_NAME,
    "strain_name": STRAIN_NAME,
    "genome_records": genome_stats["records"],
    "genome_total_bp": genome_stats["total_bp"],
    "genome_first_id": genome_stats["first_id"],
    "genome_sha256": genome_hash,
}

INPUT_FILES = {
    "uploaded_files": [str(p) for p in uploaded_paths],
    "all_detected_files": [str(p) for p in all_files],
    "genome_candidates": [str(p) for p in genome_candidates],
    "annotation_candidates": [str(p) for p in annotation_candidates],
    "protein_candidates": [str(p) for p in protein_candidates],
    "cds_candidates": [str(p) for p in cds_candidates],
    "genome_source": str(genome_source),
    "annotation_source": str(annotation_source) if annotation_source else "",
    "protein_source": str(protein_source) if protein_source else "",
    "cds_source": str(cds_source) if cds_source else "",
}

SHARP_NOTEBOOK_CONTEXT = {
    "project": SHARP_PROJECT_NAME,
    "initialized_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "job_name": RUN_NAME,
    "run_name": RUN_NAME,
    "workflow": workflow,
    "input_mode": INPUT_MODE_KEY,
    "annotation_mode": annotation_mode,
    "annotation_backend": annotation_backend,
    "paths": SHARP_PATHS,
    "input_contract": SHARP_INPUT_CONTRACT,
    "input_files": INPUT_FILES,
    "sharp_package": SHARP_PACKAGE_STATUS,
    "internal_resources": SHARP_INTERNAL_RESOURCES,
    "resource_status": SHARP_RESOURCE_STATUS,
    "hmm_press_status": SHARP_HMM_PRESS_STATUS,
    "backend": SHARP_BACKEND,
    "fimo_bin": shutil.which("fimo") or "",
    "pythonpath": os.environ.get("PYTHONPATH", ""),
    "path": os.environ.get("PATH", ""),
}

SHARP_DEPENDENCY_STATUS = {
    "ready": True,
    "sharp_package": SHARP_PACKAGE_STATUS,
    "workflow": workflow,
    "input_mode": INPUT_MODE_KEY,
    "annotation_backend": annotation_backend,
    "backend": SHARP_BACKEND,
    "resource_status": SHARP_RESOURCE_STATUS,
    "hmm_press_status": SHARP_HMM_PRESS_STATUS,
    "setup_events": SETUP_EVENTS,
}

write_json(SHARP_CONTEXT_FILE, SHARP_NOTEBOOK_CONTEXT)
write_json(INPUT_CONTRACT_FILE, SHARP_INPUT_CONTRACT)
write_json(INPUT_FILES_FILE, INPUT_FILES)
write_json(RESOURCE_MANIFEST_FILE, SHARP_RESOURCE_STATUS)
write_json(DEPENDENCY_MANIFEST_FILE, SHARP_DEPENDENCY_STATUS)
write_json(SETUP_LOG_FILE, SETUP_EVENTS)

# Aliases consumed by the execution cell (names differ from the locals above).
input_mode = INPUT_MODE_KEY
BAKTA_BIN = SHARP_BACKEND.get("bakta_bin", "")
BAKTA_DB_READY = SHARP_BACKEND.get("bakta_db_ready", False)
BAKTA_DB_PATH = SHARP_BACKEND.get("bakta_db_path", "")
BAKTA_RUNTIME_ARGS = SHARP_BACKEND.get("bakta_runtime_args", [])
BAKTA_THREADS = SHARP_BACKEND.get("bakta_threads", 0)

print("S(H)ARP input ready")
print("job:", RUN_NAME)
print("mode:", INPUT_MODE_KEY)
print("genome:", f"{genome_stats['records']} records / {genome_stats['total_bp']:,} bp")
print("sharp:", "ready")
print("rotifer:", "ready")
print("resources:", "ready")
print("annotation_backend:", annotation_backend)
print("backend:", "ready" if (not requires_bakta_execution or SHARP_BACKEND.get("bakta_db_ready")) else "check")
print("next:", "run S(H)ARP GitLab pipeline")


In [ ]:
# @title S(H)ARP — Run GitLab pipeline and export results { display-mode: "form" }
# @markdown Run S(H)ARP by calling the official GitLab `sharp.pipeline.igem_pipeline` function.

SHOW_SETUP_LOGS = True # @param {type:"boolean"}
DOWNLOAD_RESULTS_ZIP = True # @param {type:"boolean"}

from pathlib import Path
import os
import sys
import json
import time
import shutil
import zipfile
import subprocess
import importlib

os.environ["MPLBACKEND"] = "Agg"

_T0 = time.time()


def stage(message):
    """Print a timestamped progress marker (gated by SHOW_SETUP_LOGS)."""
    if SHOW_SETUP_LOGS:
        print(f"[{time.time() - _T0:6.1f}s] {message}", flush=True)


def run_logged(cmd, extra_env=None):
    """Run a subprocess; stream live when SHOW_SETUP_LOGS else capture (returned in .stdout)."""
    env = {**os.environ, "MPLBACKEND": "Agg"}
    if extra_env:
        env.update(extra_env)
    if SHOW_SETUP_LOGS:
        return subprocess.run(cmd, text=True, env=env, check=False)
    return subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          env=env, check=False)


def add_syspath(path):
    path = Path(path)
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))


def add_path(path):
    path = Path(path)
    if path.exists() and str(path) not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = str(path) + os.pathsep + os.environ.get("PATH", "")

# ============================================================
# Load context from Cell 1
# ============================================================

stage("Loading context from Cell 1.")

PROJECT_DIR = Path(globals().get("PROJECT_DIR", "/content/sharp_igem_usp_brazil_2026")).resolve()
CONFIG_DIR = Path(globals().get("CONFIG_DIR", PROJECT_DIR / "config")).resolve()

SHARP_CONTEXT_FILE = Path(globals().get("SHARP_CONTEXT_FILE", CONFIG_DIR / "sharp_notebook_context.json")).resolve()
INPUT_CONTRACT_FILE = Path(globals().get("INPUT_CONTRACT_FILE", CONFIG_DIR / "sharp_input_contract.json")).resolve()

if not SHARP_CONTEXT_FILE.exists() or not INPUT_CONTRACT_FILE.exists():
    raise RuntimeError("Missing sharp context/contract. Run Cell 1 first.")

context = json.loads(SHARP_CONTEXT_FILE.read_text(encoding="utf-8"))
contract = json.loads(INPUT_CONTRACT_FILE.read_text(encoding="utf-8"))

RUN_NAME = context.get("run_name", context.get("job_name", "sharp_run_01"))
PATHS = context.get("paths", {})

RESULTS_DIR = Path(PATHS.get("results_dir", PROJECT_DIR / "results")).resolve()
RUN_DIR = Path(PATHS.get("run_dir", RESULTS_DIR / RUN_NAME)).resolve()
LOG_DIR = Path(PATHS.get("log_dir", RUN_DIR / "logs")).resolve()
TABLE_DIR = Path(PATHS.get("table_dir", RUN_DIR / "tables")).resolve()
REPORT_DIR = Path(PATHS.get("report_dir", RUN_DIR / "report")).resolve()
ANNOTATION_DIR = RUN_DIR / "annotation"
BAKTA_RUN_DIR = RUN_DIR / "bakta"

for directory in [RUN_DIR, LOG_DIR, TABLE_DIR, REPORT_DIR, ANNOTATION_DIR, BAKTA_RUN_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

GENOME_FASTA = Path(contract["genome_fasta"]).resolve()
ANNOTATION_FILE_FROM_CONTRACT = Path(contract["annotation_file"]).resolve() if contract.get("annotation_file") else None
PROTEIN_FASTA_FROM_CONTRACT = Path(contract["protein_fasta"]).resolve() if contract.get("protein_fasta") else None
CDS_FASTA_FROM_CONTRACT = Path(contract["cds_fasta"]).resolve() if contract.get("cds_fasta") else None

input_mode = contract["mode"]
requires_bakta_execution = bool(contract.get("requires_bakta_execution", input_mode == "genome_fasta_only"))

SHARP_BACKEND = globals().get("SHARP_BACKEND", context.get("backend", {}))
SHARP_INTERNAL_RESOURCES = globals().get("SHARP_INTERNAL_RESOURCES", context.get("internal_resources", {}))

SHARP_DIR = Path(PATHS.get("sharp_dir", "/content/sharp")).resolve()
ROTIFER_LIB = Path(PATHS.get("rotifer_lib", "/content/rotifer/lib")).resolve()
ROTIFER_BIN = Path(PATHS.get("rotifer_bin", "/content/rotifer/bin")).resolve()

add_syspath(SHARP_DIR)
add_syspath(ROTIFER_LIB)
add_path(ROTIFER_BIN)

BAKTA_BIN = globals().get("BAKTA_BIN", SHARP_BACKEND.get("bakta_bin", "")) or shutil.which("bakta")
BAKTA_DB_PATH = globals().get("BAKTA_DB_PATH", SHARP_BACKEND.get("bakta_db_path", ""))
BAKTA_RUNTIME_ARGS = globals().get("BAKTA_RUNTIME_ARGS", SHARP_BACKEND.get("bakta_runtime_args", []))
BAKTA_THREADS = int(globals().get("BAKTA_THREADS", SHARP_BACKEND.get("bakta_threads", 2)) or 2)

if BAKTA_BIN:
    add_path(Path(BAKTA_BIN).parent)

# ============================================================
# Import official GitLab SHARP package
# ============================================================

stage("Importing sharp package.")

sharp = importlib.import_module("sharp")
sharp_pipeline = importlib.import_module("sharp.pipeline")
sharp_cli = importlib.import_module("sharp.cli")

if not hasattr(sharp_pipeline, "igem_pipeline"):
    raise RuntimeError("sharp.pipeline.igem_pipeline was not found.")
if requires_bakta_execution and not hasattr(sharp_cli, "run_bakta"):
    raise RuntimeError("sharp.cli.run_bakta was not found, but Bakta annotation is required.")

igem_pipeline = sharp_pipeline.igem_pipeline
run_bakta = sharp_cli.run_bakta

# ============================================================
# Resources from Cell 1
# ============================================================

heptamer_meme = Path(SHARP_INTERNAL_RESOURCES["heptamer_meme"]).resolve()
sarp_hmm = Path(SHARP_INTERNAL_RESOURCES["sarp_hmm"]).resolve()
domain_models_hmm = Path(SHARP_INTERNAL_RESOURCES["domain_models_hmm"]).resolve()
domain_modelnames = Path(SHARP_INTERNAL_RESOURCES["domain_modelnames"]).resolve()

for resource_path in [heptamer_meme, sarp_hmm, domain_models_hmm, domain_modelnames]:
    if not resource_path.exists() or resource_path.stat().st_size == 0:
        raise RuntimeError(f"Missing required resource: {resource_path}")

# ============================================================
# Prepare annotation inputs
# ============================================================

genome_nucleotide_fasta = ANNOTATION_DIR / "sharp_genome.fna"
genome_annotation = ANNOTATION_DIR / "sharp_annotation.gff3"
genome_protein_fasta = ANNOTATION_DIR / "sharp_proteins.faa"
genome_cds_fasta = ANNOTATION_DIR / "sharp_cds.ffn"

output_report = REPORT_DIR / "sharp_report.html"
ndf_table = TABLE_DIR / "sharp_neighborhoods.tsv"
fimo_table = TABLE_DIR / "sharp_fimo.tsv"
hmmscan_table = TABLE_DIR / "sharp_hmmscan.tsv"
summary_file = RUN_DIR / f"{RUN_NAME}_summary.json"
zip_path = RESULTS_DIR / f"{RUN_NAME}_sharp_results.zip"

shutil.copy2(GENOME_FASTA, genome_nucleotide_fasta)

if requires_bakta_execution:
    if not BAKTA_BIN:
        raise RuntimeError("Bakta is required but BAKTA_BIN is missing. Rerun Cell 1.")
    if not BAKTA_DB_PATH:
        raise RuntimeError("Bakta is required but BAKTA_DB_PATH is missing. Rerun Cell 1.")

    os.environ["BAKTA_DB"] = str(BAKTA_DB_PATH)

    bakta_output = BAKTA_RUN_DIR / "output"
    if bakta_output.exists():
        shutil.rmtree(bakta_output)
    bakta_output.mkdir(parents=True, exist_ok=True)

    # Drop the --threads pair from the runtime args; we pass threads explicitly.
    bakta_extra = []
    args_iter = iter(str(v) for v in BAKTA_RUNTIME_ARGS)
    for value in args_iter:
        if value == "--threads":
            next(args_iter, None)  # skip its value
            continue
        bakta_extra.append(value)

    # Same command sharp.cli.run_bakta builds, but run here with captured output
    # so a failure shows the real reason instead of a bare exit code.
    bakta_cmd = [
        str(BAKTA_BIN or "bakta"),
        "--db", str(BAKTA_DB_PATH),
        "--output", str(bakta_output),
        "--prefix", "sharp_bakta",
        "--threads", str(BAKTA_THREADS),
        "--force",
        *bakta_extra,
        str(genome_nucleotide_fasta),
    ]

    stage("Running Bakta annotation...")
    proc = run_logged(bakta_cmd)
    if proc.returncode != 0:
        if proc.stdout:  # captured mode (SHOW_SETUP_LOGS off)
            print(proc.stdout[-6000:])
        log_path = bakta_output / "sharp_bakta.log"
        if log_path.exists():
            print("---- bakta.log (tail) ----")
            print(log_path.read_text(errors="replace")[-3000:])
        raise RuntimeError(f"Bakta failed (exit {proc.returncode}). See log above.")

    # FIMO parses the annotation as GFF (sharp.pipeline.build_gff_index), so hand it the GFF3.
    genome_annotation = bakta_output / "sharp_bakta.gff3"
    genome_protein_fasta = bakta_output / "sharp_bakta.faa"
    genome_nucleotide_fasta = bakta_output / "sharp_bakta.fna"
    genome_format = "gff"

    for produced in (genome_annotation, genome_protein_fasta, genome_nucleotide_fasta):
        if not produced.exists():
            raise RuntimeError(f"Expected Bakta output missing: {produced}")

    stage("Bakta annotation done.")

else:
    if ANNOTATION_FILE_FROM_CONTRACT is None or not ANNOTATION_FILE_FROM_CONTRACT.exists():
        raise RuntimeError("Annotated genome package mode requires an annotation file.")
    if PROTEIN_FASTA_FROM_CONTRACT is None or not PROTEIN_FASTA_FROM_CONTRACT.exists():
        raise RuntimeError("Annotated genome package mode requires a protein FASTA file.")

    genome_format = contract.get("genome_format", "")
    if not genome_format:
        sfx = ANNOTATION_FILE_FROM_CONTRACT.suffix.lower()
        genome_format = "gbk" if sfx in [".gb", ".gbk", ".gbff", ".genbank"] else "gff"

    if genome_format == "gbk":
        genome_annotation = ANNOTATION_DIR / "sharp_annotation.gbff"
    else:
        genome_annotation = ANNOTATION_DIR / "sharp_annotation.gff3"
        genome_format = "gff"

    shutil.copy2(ANNOTATION_FILE_FROM_CONTRACT, genome_annotation)
    shutil.copy2(PROTEIN_FASTA_FROM_CONTRACT, genome_protein_fasta)
    if CDS_FASTA_FROM_CONTRACT and CDS_FASTA_FROM_CONTRACT.exists():
        shutil.copy2(CDS_FASTA_FROM_CONTRACT, genome_cds_fasta)

    stage("Using existing annotation package.")

# ============================================================
# Run official GitLab SHARP pipeline
# ============================================================

stage("Running igem_pipeline: FIMO + hmmscan + hmmsearch + report (long step)...")

pipeline_result = igem_pipeline(
    genome_annotation=str(genome_annotation),
    genome_format=genome_format,
    genome_protein_fasta=str(genome_protein_fasta),
    genome_nucleotide_fasta=str(genome_nucleotide_fasta),
    models_path=[str(domain_models_hmm)],
    sarp_model=str(sarp_hmm),
    return_hmmscan=True,
    after=10,
    before=10,
    run_fimo=True,
    meme_file=str(heptamer_meme),
    return_fimo=True,
    make_figure=True,
    output_report=str(output_report),
    domains_filter=str(domain_modelnames),
    filter_mode="strict",
)

if not isinstance(pipeline_result, tuple) or len(pipeline_result) != 3:
    raise RuntimeError("igem_pipeline did not return the expected tuple: (ndf, fimo, hmmscan).")

ndf, fimo, hscan = pipeline_result
stage(f"Pipeline done: {len(ndf)} neighborhood rows, {len(fimo)} FIMO, {len(hscan)} hmmscan.")

# ============================================================
# Export package outputs
# ============================================================

stage("Exporting tables and summary.")

ndf.to_csv(ndf_table, sep="\t", index=False)
fimo.to_csv(fimo_table, sep="\t", index=False)
hscan.to_csv(hmmscan_table, sep="\t", index=False)

summary = {
    "status": "completed",
    "job_name": RUN_NAME,
    "input_mode": input_mode,
    "pipeline_package": "sharp",
    "pipeline_function": "sharp.pipeline.igem_pipeline",
    "sharp_file": str(Path(sharp.__file__).resolve()),
    "pipeline_file": str(Path(sharp_pipeline.__file__).resolve()),
    "genome_annotation": str(genome_annotation),
    "genome_format": genome_format,
    "genome_protein_fasta": str(genome_protein_fasta),
    "genome_nucleotide_fasta": str(genome_nucleotide_fasta),
    "models_path": [str(domain_models_hmm)],
    "sarp_model": str(sarp_hmm),
    "meme_file": str(heptamer_meme),
    "domains_filter": str(domain_modelnames),
    "ndf_rows": int(len(ndf)),
    "fimo_rows": int(len(fimo)),
    "hmmscan_rows": int(len(hscan)),
    "outputs": {
        "neighborhood_table": str(ndf_table),
        "fimo_table": str(fimo_table),
        "hmmscan_table": str(hmmscan_table),
        "html_report": str(output_report) if output_report.exists() else "",
        "zip": str(zip_path),
    },
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}

summary_file.write_text(json.dumps(summary, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

context.update({
    "pipeline_completed": True,
    "pipeline_package": "sharp",
    "pipeline_function": "sharp.pipeline.igem_pipeline",
    "summary_file": str(summary_file),
    "zip_path": str(zip_path),
    "html_report": str(output_report) if output_report.exists() else "",
    "summary": summary,
})
SHARP_CONTEXT_FILE.write_text(json.dumps(context, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

stage("Zipping results.")

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(RUN_DIR))

# Aliases for downstream use (names differ from the locals above).
NDF, FIMO, HMSCAN = ndf, fimo, hscan
NDF_TABLE, FIMO_TABLE, HMSCAN_TABLE = ndf_table, fimo_table, hmmscan_table
HTML_REPORT = output_report if output_report.exists() else None
SUMMARY_FILE = summary_file
ZIP_PATH = zip_path
SHARP_ANALYSIS_SUMMARY = summary

stage("S(H)ARP GitLab pipeline complete.")
print("ndf_rows:", len(ndf))
print("fimo_rows:", len(fimo))
print("hmmscan_rows:", len(hscan))
print("html_report:", output_report if output_report.exists() else "")
print("zip:", zip_path)

if DOWNLOAD_RESULTS_ZIP:
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception:
        pass
